<a href="https://colab.research.google.com/github/di-yeferson/analitica-empresarial-integrada/blob/main/LC1_AEI_QUISPE%2C%20DIEGO_2026_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso: Andes Supply S.A.C.

Andes Supply S.A.C. es una empresa peruana que provee bienes y servicios industriales a las compañías mineras del país: repuestos, mantenimiento de equipo pesado y servicios de logística en mina. Vende a crédito y actualmente otorga a todos sus clientes las mismas condiciones: noventa días para pagar.

En 2025 dos clientes dejaron facturas impagas y la gerencia general decidió que, a partir de 2026, las condiciones de crédito dejarán de ser iguales para todos. El área comercial debe clasificar a cada cliente minero en uno de tres tramos: crédito a 90 días, crédito a 30 días o pago adelantado. Además, debe sustentar la clasificación ante el cliente.

Andes Supply no cuenta con información interna de sus clientes. Solo puede acceder a la información financiera pública que las mineras presentan ante la Superintendencia del Mercado de Valores.

**Pregunta de negocio:** ¿qué condiciones de crédito debe otorgar Andes Supply a cada cliente minero, con qué evidencia lo sustenta y qué no puede afirmar con la información que tiene?

Los estados financieros de las mineras y las cotizaciones del BCRP son datos reales, oficiales y públicos. Andes Supply es una empresa ficticia creada únicamente para contextualizar la decisión.

# Ejercicio 1 — Ficha de trazabilidad del activo de datos (2 puntos)

**Resultado exigido.** Documente el alcance real del conjunto de datos que acaba de descargar. La gerencia debe poder saber, sin abrir el código, de dónde proviene la información y hasta dónde llega.

**La entrega debe contener:**

- La fuente y el servicio oficial efectivamente consultados, y el ejercicio económico al que corresponden los estados financieros.
- La cantidad de empresas distintas que devuelve la consulta de información financiera.
- La cantidad de sectores económicos distintos presentes en esa respuesta.
- La cantidad de cuentas contables distintas que trae el estado de situación financiera.
- Las monedas en que reportan las empresas del conjunto.

**Criterio de aceptación.** Los cinco valores numéricos deben obtenerse por cálculo sobre las tablas descargadas. Si alguno aparece escrito literalmente en el código, el criterio se considera no logrado.

In [ ]:
%pip install -q polars==1.17.1 duckdb==1.1.3

import json, html, re
from xml.etree import ElementTree as ET
import requests
import polars as pl
import pandas as pd          # solo para recibir la respuesta de la SMV
import plotly.express as px
from IPython.display import display

pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_width_chars(180)

SERVICIO_SMV = "https://mvnet.smv.gob.pe/ws_od_eeff/WebServiceInfoFinanciera.asmx"
EJERCICIO = 2024
SECTOR = "MINERAS"

# Codigos oficiales de cuenta del Estado de Situacion Financiera.
# Se identifican por CODIGO y no por descripcion: existen cuentas cuyo texto
# tambien contiene "Activos Corrientes" sin ser el total.
TOTAL_ACTIVO_CORRIENTE = "1D01ST"
TOTAL_PASIVO_CORRIENTE = "1D03ST"

print("polars:", pl.__version__)
print("Entorno listo.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 39.5 MB/s eta 0:00:00
polars: 1.17.1
Entorno listo.


In [ ]:
def _local(etiqueta):
    return etiqueta.split("}")[-1]

def _a_dataframe(payload):
    payload = html.unescape(payload).strip()
    try:
        objeto = json.loads(payload)
        if isinstance(objeto, dict): objeto = objeto.get("rows", objeto.get("data", objeto))
        if isinstance(objeto, dict): objeto = [objeto]
        return pd.DataFrame(objeto)
    except json.JSONDecodeError:
        raiz = ET.fromstring(payload); registros = []
        for nodo in raiz.iter():
            hijos = list(nodo)
            if len(hijos) >= 5 and all(not list(h) for h in hijos):
                registros.append({_local(h.tag): h.text for h in hijos})
        return pd.DataFrame(registros)

def descargar_smv(operacion, ejercicio, periodo="A", tipo="I"):
    sobre = f"""<?xml version="1.0" encoding="utf-8"?>
    <soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
      <soap:Body><{operacion} xmlns="http://tempuri.org/">
        <Ejercicio>{ejercicio}</Ejercicio><Periodo>{periodo}</Periodo><Tipo>{tipo}</Tipo>
      </{operacion}></soap:Body></soap:Envelope>"""
    respuesta = requests.post(
        SERVICIO_SMV, data=sobre.encode("utf-8"), timeout=180,
        headers={"Content-Type": "text/xml; charset=utf-8",
                 "SOAPAction": f'"http://tempuri.org/{operacion}"'})
    respuesta.raise_for_status()
    raiz = ET.fromstring(respuesta.content)
    resultado = next((n for n in raiz.iter() if _local(n.tag) == f"{operacion}Result"), None)
    if resultado is None or not (resultado.text or "").strip():
        raise ValueError(f"La SMV no devolvio datos para {operacion}.")
    return pl.from_pandas(_a_dataframe(resultado.text))

principales = descargar_smv("obtener_InfoFinanciera", EJERCICIO)
balance = descargar_smv("obtener_BalanceGeneral", EJERCICIO)

print("Cuentas principales      :", principales.shape)
print("Estado de situacion fin. :", balance.shape)
display(principales.head(3))

Cuentas principales      : (276, 16)
Estado de situacion fin. : (20574, 15)


RPJ,TipoEmpresa,TipoSector,NombreEmpresa,RUC,CIIU,Ejercicio,TipoInformacion,Trimestre,Moneda,MetodoFlujoEfectivo,ActivoTotal,PatrimonioTotal,TotalIngreso,UtilidadNeta,PasivoTotal
str,str,str,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64
"""L00474""","""EMPRESAS MERCADO ALTERNATIVO D…","""DIVERSOS""","""A. JAIME ROJAS REPRESENTACIONE…","""20102032951""","""5190""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Directo""",77710,39497,77196,6786,38213
"""I00004""","""SOCIEDADES ADMINISTRADORAS DE …","""""","""AC CAPITALES SOCIEDAD ADMINIST…","""20504893295""","""6430""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Directo""",9070,7373,4880,-85,1697
"""OE7511""","""SOCIEDADES ADMINISTRADORAS DE …","""""","""ACRES SOCIEDAD ADMINISTRADORA …","""20601498996""","""6430""","""2024""","""Anual Individual""","""Anual""","""Soles""","""M todo Indirecto""",2724,2238,2348,176,486


In [ ]:
# SOLUCION | Ejercicio 1 (2 puntos) — Ficha de trazabilidad
# Los cinco valores se obtienen por calculo sobre las tablas descargadas.
# Ninguno esta escrito a mano: ese es el criterio de aceptacion del ejercicio.
FICHA = {
    "fuente":             "SMV - Portal de Datos Abiertos",
    "servicio":           SERVICIO_SMV,
    "ejercicio":          EJERCICIO,
    "empresas_totales":   principales["NombreEmpresa"].n_unique(),
    "sectores_distintos": principales["TipoSector"].n_unique(),
    "cuentas_balance":    balance["Cuenta"].n_unique(),
    "monedas":            sorted(principales["Moneda"].unique().to_list()),
}

for k, v in FICHA.items():
    print(f"{k:>20}: {v}")

# Error frecuente: contar filas en lugar de valores distintos. `principales`
# trae varias filas por empresa (una por tipo de informacion), de modo que
# principales.height NO es la cantidad de empresas.
print("\nFilas de principales      :", principales.height)
print("Empresas distintas        :", principales["NombreEmpresa"].n_unique())


              fuente: SMV - Portal de Datos Abiertos
            servicio: https://mvnet.smv.gob.pe/ws_od_eeff/WebServiceInfoFinanciera.asmx
           ejercicio: 2024
    empresas_totales: 276
  sectores_distintos: 10
     cuentas_balance: 482
             monedas: ['D lares', 'Soles']

Filas de principales      : 276
Empresas distintas        : 276


# Ejercicio 2 — Delimitación del sector evaluable (2 puntos)

**Resultado exigido.** Obtenga la tabla de empresas mineras sobre la que se construirá todo el análisis posterior. No todas las empresas que devuelve el servicio son analizables: algunas pertenecen a otros sectores, otras reportan periodos que no son anuales y otras presentan cuentas incompletas.

**La entrega debe contener:**

- Solo empresas del sector minero.
- Solo información de periodicidad anual.
- Las cinco magnitudes del análisis —activo total, patrimonio, ingresos, utilidad neta y pasivo total— expresadas como valores numéricos.
- Sin empresas que presenten alguna de esas cinco magnitudes vacía.
- Sin empresas con ingresos iguales a cero, porque impiden calcular el margen.

**Criterio de aceptación.** El notebook debe informar cuántas empresas quedaron disponibles tras la delimitación. Una tabla que conserve el total de empresas descargadas indica que los criterios no se aplicaron.

# Ejercicio 3 — Perfil financiero comparado del sector (2 puntos)

**Resultado exigido.** Construya el perfil que permitirá comparar a los clientes entre sí. Se exigen seis indicadores por empresa: margen neto, rentabilidad sobre activos, rentabilidad sobre patrimonio, razón corriente, razón de endeudamiento y apalancamiento.

**La entrega debe contener:**

- El activo corriente y el pasivo corriente deben extraerse del estado de situación financiera identificando la cuenta por su código oficial y no por su descripción textual.
- Los seis indicadores deben quedar incorporados como columnas del perfil, no impresos sueltos.
- El resultado debe presentarse ordenado y legible, con el nombre de la empresa y su moneda.

**Criterio de aceptación.** Ninguna razón puede resultar igual a cero ni infinita. Si la razón corriente sale cero, la cuenta seleccionada no es la correcta: existen cuentas cuya descripción contiene el texto «Activos Corrientes» sin ser el total, y valen cero.

# Ejercicio 4 — Lectura crítica del ranking (2 puntos)

**Resultado exigido.** Identifique a la empresa que encabeza el sector por rentabilidad sobre patrimonio y extraiga, para esa misma empresa, su endeudamiento, su apalancamiento, su patrimonio y su pasivo total.

**La entrega debe contener:**

- La empresa debe quedar determinada por el propio cálculo sobre el perfil.
- Las cuatro magnitudes deben corresponder a la empresa identificada, obtenidas de la misma tabla.

**Criterio de aceptación.** El nombre de la empresa no puede aparecer escrito en el código. Si el sector cambiara de composición, la respuesta debería actualizarse sola al reejecutar.

# Ejercicio 5 — Política de crédito de Andes Supply (2 puntos)

**Resultado exigido.** Clasifique a cada cliente minero en uno de tres tramos de condiciones comerciales: crédito a noventa días, crédito a treinta días o pago adelantado. Esta es la decisión que la gerencia va a aplicar.

**La entrega debe contener:**

- Los umbrales que separan los tramos los define su equipo, pero deben aparecer declarados como valores explícitos y localizables, no incrustados dentro de la lógica.
- La clasificación debe emplear al menos dos indicadores distintos; un solo indicador no sustenta una política de crédito.
- El resultado debe mostrar la clasificación por empresa y el recuento de clientes en cada tramo.

**Criterio de aceptación.** Cada umbral debe poder justificarse ante un cliente que reclame su clasificación. Umbrales elegidos sin criterio explicable se califican como no logrados aunque el código funcione.

# Ejercicio 6 — Consulta reproducible de la cartera en SQL (2 puntos)

**Resultado exigido.** La cartera resultante debe poder consultarse por personas que leen SQL y no Python. Formule sobre el perfil una consulta que devuelva las empresas con razón corriente igual o mayor que uno y endeudamiento inferior a 0,60, ordenadas de mayor a menor rentabilidad sobre patrimonio.

**La entrega debe contener:**

- Al menos cuatro columnas en la selección.
- Dos condiciones de filtrado enlazadas entre sí.
- Una columna calculada que etiquete el nivel de riesgo según las condiciones que su equipo defina.
- Ordenamiento explícito del resultado.

**Criterio de aceptación.** El resultado debe contrastarse con la clasificación construida en el ejercicio anterior. La coincidencia o discrepancia entre ambos caminos es lo que se interpreta en la pregunta escrita 6.1.

# Preguntas escritas de interpretación

Las preguntas escritas no otorgan puntaje separado: son la evidencia con la que se califica la interpretación dentro de cada criterio de la rúbrica. Un notebook que ejecuta correctamente pero no interpreta no alcanza el nivel Excelente en ningún criterio.

El criterio «Escalera analítica» se evalúa únicamente con la pregunta escrita 2.1, que no tiene ejercicio de código asociado.

## Pregunta 1.1 — Qué preguntas de negocio permite y no permite responder el conjunto de datos

**Sí permite:** identificar empresas con utilidad neta negativa en 2024; comparar rentabilidad, liquidez y estructura de financiamiento; estimar qué empresa tiene mayor capacidad contable de cubrir pasivos corrientes; y medir qué empresa depende más del financiamiento de terceros.

**No permite:** saber cuáles empresas son realmente clientes de Andes Supply, cuánto compró cada una, si paga tarde, si incumplirá en el futuro o si tiene intención de pagar. Para responder eso se necesita el historial interno de ventas, facturas, vencimientos, pagos, mora, garantías y exposición crediticia por cliente.

## Pregunta 2.1 — Clasificación de seis preguntas en los escalones de la escalera analítica

| Pregunta | Escalón | Justificación |
|---|---|---|
| ¿Cuál fue la utilidad neta de cada cliente en 2024? | Descriptiva | Resume un resultado ya observado. |
| ¿Por qué cayó la rentabilidad de los clientes de cobre? | Diagnóstica | Busca explicar las causas del resultado. |
| ¿Qué probabilidad hay de que este cliente deje de pagar? | Predictiva | Estima la probabilidad de un evento futuro. |
| ¿A qué clientes debo exigir pago adelantado? | Prescriptiva | Recomienda una acción comercial. |
| ¿Cuánto endeudamiento tiene cada cliente? | Descriptiva | Mide una situación financiera observada. |
| ¿Qué pasaría con mi cartera si el cobre cae 20 %? | Predictiva | Proyecta el resultado bajo un escenario futuro. |

El laboratorio trabaja directamente en el escalón descriptivo y formula una regla prescriptiva preliminar. No demuestra causas ni genera una predicción validada porque solo dispone de un corte financiero, no de historial de pagos ni de una variable objetivo de incumplimiento.

## Pregunta 3.1 — Qué comparaciones invalida la convivencia de dos monedas en el sector

No es válido comparar directamente el ingreso de Cerro Verde con el de Shougang ni sumar los activos de todas las empresas, porque los montos mezclan dólares y soles. Esas operaciones requieren convertir primero a una moneda común con un criterio de tipo de cambio consistente. Sí pueden compararse el margen neto y la razón corriente entre empresas de monedas distintas, porque son razones adimensionales calculadas con importes expresados en la misma moneda dentro de cada empresa.

## Pregunta 4.1 — Por qué el ROE más alto del sector no identifica al cliente más sólido



## Código de apoyo para la pregunta 5.1 — Cotizaciones oficiales del BCRP

In [ ]:
SERIES = {"PN01652XM": "Cobre (cUS$/lb)", "PN01654XM": "Oro (US$/oz)",
          "PN01653XM": "Estanio (cUS$/lb)", "PN01655XM": "Plata (US$/oz)"}
BASE_BCRP = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api"



Observaciones traidas del BCRP: 96


serie,ene_2023,dic_2024,var_pct
str,f64,f64,f64
"""Oro (US$/oz)""",1894.068182,2638.559091,39.3
"""Plata (US$/oz)""",23.752636,30.466636,28.3
"""Estanio (cUS$/lb)""",1270.040595,1308.213486,3.0
"""Cobre (cUS$/lb)""",406.296088,404.30647,-0.5


## Pregunta 5.1 — Relación entre la cotización de metales y el resultado de dos empresas, sin afirmar causalidad



## Pregunta 6.1 — Contraste entre la cartera obtenida con SQL y la clasificación construida con Polars



# Informe ejecutivo A a F

## A. Política recomendada

## B. Explicación al cliente


## C. Estadio de madurez de Andes Supply


## D. Modelo DELTA Plus en siete dimensiones



## E. Límites del análisis


## F. Iniciativa priorizada

